In [1]:
## importing libraries
import os
from dotenv import load_dotenv
load_dotenv()

from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence.aio import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentAnalysisFeature
from azure.ai.documentintelligence.models import DocumentContentFormat
from openai import AsyncAzureOpenAI
import base64




In [2]:
file_path = r"C:\Users\SumeetMaheshwari\Desktop\workspace\doc-comparator\sample\Output 1.pdf"

# Load environment variables
endpoint = os.getenv("DOCUMENT_INTELLIGENCE_ENDPOINT")
key = os.getenv("DOCUMENT_INTELLIGENCE_KEY")

In [3]:
### function for the reading the pdf as input and return the result
document_intelligence_client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

async def data_extractor(pdf_path:str):
    with open(pdf_path, "rb") as f:
        pdf_bytes = f.read()

    # Base64 encode PDF
    base64_encoded_pdf = base64.b64encode(pdf_bytes).decode("utf-8")

    analyze_request = {
        "base64Source": base64_encoded_pdf
    }

    # Start analysis
    poller = await document_intelligence_client.begin_analyze_document(
        "prebuilt-layout",
        analyze_request,
        output_content_format=DocumentContentFormat.MARKDOWN
        
    )
    
    result = await poller.result()
    page_wise_md = result.content.split("<!-- PageBreak -->")

    page_wise_ocr = []

    for page_idx, page in enumerate(result.pages):
        page_wise_context = ""
        if not(page.lines):
            import pdb;pdb.set_trace()
        for line_idx, line in enumerate(page.lines):
            page_wise_context += line.content

        page_wise_ocr.append(page_wise_context)

    return page_wise_md, page_wise_ocr


In [4]:
page_wise_md, page_wise_ocr = await data_extractor(file_path)

In [30]:
with open(
    r"C:\Users\SumeetMaheshwari\Desktop\workspace\doc-comparsion-using-Azure-OCR\Backend\artifact\page_wise_md_formated.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(str(page_wise_md))

In [32]:
with open(
    r"C:\Users\SumeetMaheshwari\Desktop\workspace\doc-comparsion-using-Azure-OCR\Backend\artifact\page_wise_ocr.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(str(page_wise_ocr))

In [5]:
print(page_wise_md)

['Master Batch Record\n30000773 - ARIPIPRAZOLE 5 MG TAB\n\nEffective\n\n\n<figure>\n\nAlembic\nTouching Lives over"\n100\nyears\n\n</figure>\n\n\n<table>\n<tr>\n<td>ID/Version/Description</td>\n<td>F1M00332/00000001/Aripiprazole Tab.USP 5mg</td>\n</tr>\n<tr>\n<td>Standard batch size</td>\n<td>1,500,000 NO</td>\n</tr>\n<tr>\n<td>Estimated yield</td>\n<td>1,500,000 NO</td>\n</tr>\n</table>\n\n\nLong description\nProduct Name : Aripiprazole Tablets USP 5 mg\nGeneric Name of Product : Aripiprazole Tablets USP\nBrand Name : NA\nStrength : 5 mg\nStandard Batch Size in Kg : 142.500 kg\nStandard Batch Size in Unit : 1,500,000 Tablets\nReference Document No. : MFC/0353-05\n\nMarket/Customer : Export /US\nRef. BMR No. : F1\\BMR\\00837 & 3.0\n\n\n<table>\n<tr>\n<th colspan="4">Approval</th>\n</tr>\n<tr>\n<th colspan="2">Status</th>\n<th colspan="2">Text</th>\n</tr>\n<tr>\n<td colspan="4">Master Batch Record 10#F1M00332#Aripiprazole Tab.USP 5mg#30000773# --- ##00000001</td>\n</tr>\n<tr>\n<td rowsp

In [6]:
print(page_wise_ocr)

['Master Batch Record30000773 - ARIPIPRAZOLE 5 MG TABEffectiveAlembicTouching Lives over"100yearsID/Version/DescriptionF1M00332/00000001/Aripiprazole Tab.USP 5mgStandard batch size1,500,000 NOEstimated yield1,500,000 NOLong descriptionProduct Name : Aripiprazole Tablets USP 5 mgGeneric Name of Product : Aripiprazole Tablets USPBrand Name : NAStrength : 5 mgStandard Batch Size in Kg : 142.500 kgStandard Batch Size in Unit : 1,500,000 TabletsReference Document No. : MFC/0353-05Market/Customer : Export /USRef. BMR No. : F1\\BMR\\00837 & 3.0ApprovalStatusTextMaster Batch Record 10#F1M00332#Aripiprazole Tab.USP 5mg#30000773# --- ##00000001DraftObject was createdDate - User: 24/09/2025 15:57:34 - 27759 / Kinjal MehtaIn circulationPut MBR in circulationDate - User: 24/09/2025 16:30:55 - 27759 / Kinjal MehtaIn circulationMBR approval by technical support teamDate - User: 30/09/2025 12:26:34 - 16328 / Yesha PatelIn circulationMBR approval by production teamDate - User: 07/10/2025 14:08:16 - 918

In [7]:
data_extracted_without_tags = "\n\n".join(page_wise_ocr[:])

In [8]:
len(data_extracted_without_tags)


398384

In [9]:
# print(data_extracted_without_tags)

In [10]:
import re
from collections import Counter


In [11]:
# def normalize_line(line: str) -> str:
#     return re.sub(r"\s+", " ", line.strip())


In [12]:
# def detect_repeating_lines(text: str, min_repeat_ratio=0.2):
#     lines = [normalize_line(l) for l in text.split("\n") if l.strip()]
#     total_pages_est = max(1, text.count("PAGE"))

#     freq = Counter(lines)

#     repeating = {
#         line for line, count in freq.items()
#         if count >= max(2, int(total_pages_est * min_repeat_ratio))
#     }

#     return repeating


In [13]:
# HEADER_PATTERNS = [
#     r"^MASTER BATCH RECORD\d+\s*-",          # Master Batch Record30000773 -
#     r"^EFFECTIVE\s+ALEMBIC",                 # EffectiveAlembic...
#     r"^ALEMBIC\s+TOUCHING\s+LIVES",           # Branding line
#     r"^ID/VERSION/DESCRIPTION",               # Header metadata line
# ]

# FOOTER_PATTERNS = [
#     r"^PAGE\s+\d+\s+OF\s+\d+$",
#     r"^\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2}:\d{2}$",
#     r"^V\d+$",
#     r"^[A-Z][a-z]+\s+[A-Z][a-z]+$",           # Nishendu Nadpara
# ]



In [14]:
# def remove_headers_and_footers(text: str) -> str:
#     cleaned = []

#     for line in text.split("\n"):
#         norm = normalize_line(line)

#         if not norm:
#             cleaned.append("")  # preserve paragraph spacing
#             continue

#         # Header check
#         if any(re.search(p, norm, re.I) for p in HEADER_PATTERNS):
#             continue

#         # Footer check
#         if any(re.search(p, norm, re.I) for p in FOOTER_PATTERNS):
#             continue

#         cleaned.append(line)

#     return "\n".join(cleaned)


In [15]:
# clean_text = remove_headers_and_footers(data_extracted_without_tags)

# for i, line in enumerate(clean_text.split("\n")[:200]):
#     print(f"{i:03d}: {line}")


In [16]:
# import re

# def reconstruct_lines(flat_text: str) -> str:
#     # Insert newline before Page markers
#     text = re.sub(r"(Page\s+\d+\s+of\s+\d+)", r"\n\1\n", flat_text, flags=re.I)

#     # Insert newline before BO IDs
#     text = re.sub(r"(BO ID\s*-\s*BO Description:)", r"\n\1", text, flags=re.I)

#     # Insert newline before F1M blocks
#     text = re.sub(r"(F1M\d{5}/)", r"\n\1", text)

#     # Insert newline before parent headings
#     text = re.sub(
#         r"(APPROVAL|BASIC OPERATIONS|DOCUMENTS|SUMMING BILL OF TARGET MATERIAL|PRODUCTION UNIT|DETAILED BO)",
#         r"\n\1",
#         text,
#         flags=re.I
#     )

#     # Normalize excessive newlines
#     text = re.sub(r"\n{2,}", "\n", text)

#     return text.strip()


In [17]:
# reconstructed = reconstruct_lines(data_extracted_without_tags)

# for i, line in enumerate(reconstructed.split("\n")):
#     print(f"{i:03d}: {line}")


In [18]:
# clean_text = remove_headers_and_footers(reconstructed)
# for i, line in enumerate(clean_text.split("\n")):
#     print(f"{i:03d}: {line}")

In [19]:
# parents = extract_parent_chunks(clean_text)

# print("Parents found:", len(parents))
# for p in parents:
#     print("\n====", p["title"], "====")
#     print(p["text"][:300])


In [20]:
PARENT_PATTERN = re.compile(
    r"(?im)^\s*(APPROVAL|BASIC OPERATIONS|DOCUMENTS|SUMMING BILL.*|PRODUCTION UNIT|DETAILED BO.*)\s*$"
)


def extract_parent_chunks(clean_text: str):
    matches = list(PARENT_PATTERN.finditer(clean_text))

    parents = []

    for i, match in enumerate(matches):
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(clean_text)

        parents.append({
            "parent_title": match.group(1).strip(),
            "text": clean_text[start:end].strip()
        })

    return parents


In [28]:
with open(
    r"C:\Users\SumeetMaheshwari\Desktop\workspace\doc-comparsion-using-Azure-OCR\Backend\artifact\data.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(data_extracted_without_tags)


In [21]:
parents = extract_parent_chunks(data_extracted_without_tags)

print("Parents found:", len(parents))
for p in parents:
    print("\n====", p["parent_title"], "====")
    print(p["text"])


Parents found: 0
